In [ ]:
import pandas as pd
import numpy as np


# Loading the dataset

In [ ]:
df = pd.read_csv(r'D:\retail_analytics\data\online_retail_II.csv')
print(df.head(5))

In [ ]:
def summary(ds):
    final_count = pd.DataFrame({
        "Column": ds.columns,
        "Data Types": ds.dtypes.values,
        "Non Null Counts": ds.notnull().sum().values,
        "Null Count": ds.isnull().sum().values,
        "Null Percent": round((ds.isnull().sum() / len(ds))*100,2).values,
        "Unique values": ds.nunique().values
    })

    print("=" * 60)
    print("DATASET OVERVIEW")
    print("=" * 60)
    print(f"Rows: {df.shape[0]}")
    print(f"Columns: {df.shape[1]}")
    print(f"Numeric Columns: {len(df.select_dtypes(include='number').columns)}")
    print(f"Categorical Columns: {len(df.select_dtypes(exclude='number').columns)}")

    return final_count

print(summary(df))

In [ ]:
df.describe()

# Data Cleaning

In [ ]:
df = df.dropna(subset=['Customer ID'])

### Observation:
- Over here we can notice from the summary that we have 22% of the values without customer ID which is a valid information.
- And since we cannot fill those values as they are unique, we have to drop them
- And after dropping them if we again see that summary, we notice that the feature 'Description''s null value are also gone. That means there were overlapping null values present.
- Customer ID is essential for downstream customer-level analysis (like RFM segmentation in BigQuery later), which is why dropping instead of filling is the right call here. 

In [ ]:
df[df['Invoice'].str.startswith('C')]['Invoice'].unique()

In [ ]:
df = df[~df['Invoice'].str.startswith('C', na=False)]

In [ ]:
df[df['Invoice'].str.startswith('C')]

### Observation
- Now mentioned in the documentaiton of the dataset, there are values in the invoice starting with 'C' and this are the values where the orders is either cancelled or returned.
- So including such rows in our analysis would skew revenue and product metrics, hence we have removed them from the dataset.

In [ ]:
(df['Price'] == 0).sum()

In [ ]:
df = df[df['Price'] != 0]

### Observation
- If you notice in the describe() output above, the minimum value for Price was 0. It is not possible to sell something for free, hence those rows are likely data entry errors and have been removed from the dataset.

In [ ]:
df.describe()

In [ ]:
print(summary(df))

# Feature Engineering

In [ ]:
df.head(5)

In [ ]:
df['Revenue'] = np.round(df['Price'] * df['Quantity'], 2)

In [ ]:
df.head(5)

In [ ]:
df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'])
print(df['InvoiceDate'].dtype)

In [ ]:
df['year'] = df['InvoiceDate'].dt.year
print(df['year'].head(5))

In [ ]:
df['month'] = df['InvoiceDate'].dt.month
print(df['month'].head(5))

In [ ]:
df['hour'] = df['InvoiceDate'].dt.hour
print(df['hour'])

In [ ]:
df['day'] = df['InvoiceDate'].dt.day_name()
print(df['day'].head(5))

In [ ]:
print(df.head(5))

### Observation
- Here we had to calculate the total revenue earned each day that is by multiplying quantity and price so that we get the revenue and this is the most important feature for anaylsis
- Then we had to convert the invoiceDate from object to datetime data type.
- Once that is done, we have to split that feature into year, month, hours and day of the week so that when we are answering question like which day or month or year has the highest sales, we can do it easily.

In [ ]:
df.to_csv(r'D:\retail_analytics\data\retail_cleaned.csv', index=False)

In [ ]:
summary(df)

In [ ]:
new_df = pd.read_csv(r'D:\retail_analytics\data\retail_cleaned.csv')
new_df.head(5)

# Creating Tables In Python That Matches With MS SQL Server

# Creating Dim_date

In [ ]:
dim_date = new_df[['InvoiceDate', 'year', 'month', 'day', 'hour']].drop_duplicates().reset_index(drop=True)
dim_date.insert(0, 'DateID', range(1, len(dim_date)+1))
print(dim_date.head())

- This is the first table we have created in the ms sql and now in python we are creating a data frame where we are selecting columns from our main dataset
- over here we should keep in mind that we need to keep the column names same as that what we had created tables in the ms sql server
- if the column names does not match, we will get error as column name does not match 

# Creating Dim_customer

In [ ]:
dim_customer = new_df[['Customer ID', 'Country']].drop_duplicates(subset=['Customer ID']).reset_index(drop=True)
dim_customer = dim_customer.rename(columns={'Customer ID': 'CustomerID'})
print(dim_customer)

# Creating Din_product

In [ ]:
dim_product = new_df[['StockCode', 'Description']].drop_duplicates(subset=['StockCode']).reset_index(drop=True)
print(dim_product.head())

# Creating Fact_sales table

In [ ]:
new_df = new_df.merge(dim_date[['DateID', 'InvoiceDate']], on='InvoiceDate')

In [ ]:
fact_sales = new_df[['Invoice','Customer ID','StockCode','DateID','Quantity','Price','Revenue']]
fact_sales = fact_sales.rename(columns={'Customer ID': 'CustomerID', 'Invoice':'InvoiceNo'})
print(fact_sales.head())

- Now in this fact_sales, we have to add the 'date_id' column which was not there in the main dataset so we have to merge it from dim_date based on invoicedate
- If you notice, I have renamed some columns because the names did not match with the columns i had created in the ms sql server so i have changed the names to match it.

# Connecting Python with MS SQL

In [ ]:
from sqlalchemy import create_engine

engine = create_engine('mssql+pyodbc://akshat/retail_analytics?driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes')

In [ ]:
print(engine)

- To connect python with ms sql server, we use libraires sqlalchemy and pyodbc.
- In sqlalchemy, we have create_engine function that helps us connect the two and this string we dont have to remember it it can be taken from google search.
- in that string we just need to change the name of the server, databasename and since it is system authenticated i have not used password and username i have used trusted_connection = yes

# Populating values of dim_customer from python to MS SQL

In [ ]:
dim_customer.to_sql('dim_customer', con=engine, if_exists='append', index=False)

In [ ]:
dim_customer.shape

# Populating dim_date from python to MS SQL

In [ ]:
dim_date.to_sql('dim_date', con = engine, if_exists = 'append', index = False)

In [ ]:
dim_date.shape

# Populating dim_product from python to MS SQL

In [ ]:
dim_product.to_sql('dim_product', con = engine, if_exists = 'append', index = False)

In [ ]:
dim_product.shape

# Populating fact_sales from python to MS SQL

In [ ]:
fact_sales.to_sql('fact_sales', con=engine, if_exists= 'append', index= False)

In [ ]:
fact_sales.shape

# Big Query

In [ ]:
from google.cloud import bigquery

In [ ]:
client = bigquery.Client.from_service_account_json('bq-learning-497909-8bee4c027065.json')
print(client.project)

In [ ]:
from google.oauth2 import service_account

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()
credentials_path = os.getenv('BQ_CREDENTIALS_PATH')
credentials = service_account.Credentials.from_service_account_file(credentials_path)

In [ ]:
dim_customer.to_gbq('retail_data.dim_customer','bq-learning-497909', if_exists='replace', credentials=credentials)

In [ ]:
dim_product.to_gbq('retail_data.dim_product', 'bq-learning-497909', if_exists='replace', credentials=credentials)

In [ ]:
dim_date.to_gbq('retail_data.dim_date','bq-learning-497909',if_exists='replace',credentials=credentials)

In [ ]:
fact_sales.to_gbq('retail_data.fact_sales','bq-learning-497909',if_exists='replace',credentials=credentials)

In [ ]:
# Which country generates the most revenue
query = """
select customer.Country, sum(sales.Revenue) as Total_revenue from `bq-learning-497909.retail_data.dim_customer` as customer
inner join `bq-learning-497909.retail_data.fact_sales` as sales ON
customer.CustomerID = sales.CustomerID
group by customer.Country
order by sum(sales.Revenue) desc
"""

result = client.query(query).to_dataframe()
result['Total_revenue'] = result['Total_revenue'].round(2)
print(result.to_string())

In [ ]:
# -- What are the top 10 best selling products by revenue?
query = """
select product.StockCode, product.Description ,sum(sales.Revenue) Total_Revenue from `bq-learning-497909.retail_data.dim_product` as product
inner join `bq-learning-497909.retail_data.fact_sales` as sales ON
product.StockCode = sales.StockCode
group by product.StockCode, product.Description
order by sum(sales.Revenue) desc
limit 10;
"""

result = client.query(query).to_dataframe()
print(result)

In [ ]:
# -- What is the total revenue per month?
query = """
select Order_date.month, Order_date.year, SUM(sales.Revenue) as Net_sales from `bq-learning-497909.retail_data.dim_date` as Order_date
inner join `bq-learning-497909.retail_data.fact_sales` as sales ON
Order_date.DateID = sales.DateID
group by Order_date.month, Order_date.year
order by Order_date.year, Order_date.month
"""

result = client.query(query).to_dataframe()
print(result)

In [ ]:
# -- Which day of the week has the highest total revenue?

query = """
select Order_date.day, SUM(sales.Revenue) as Net_sales from `bq-learning-497909.retail_data.dim_date` as Order_date
inner join `bq-learning-497909.retail_data.fact_sales` as sales ON
Order_date.DateID = sales.DateID
group by Order_date.day
order by SUM(sales.Revenue) desc
"""

result = client.query(query).to_dataframe()
result['Net_sales'] = result['Net_sales'].round(2)
print(result)

# MS SQL V/S BIG QUERY:
 - MS SQL Server was used to:
    - Store data in a proper relational star schema
    - Practice data modeling (primary keys, foreign keys, dimensions, facts)
    - Simulate an operational database like companies use for day-to-day transactions

- BigQuery was used to:
   - Simulate a cloud analytical warehouse
   - Run large-scale analytical SQL queries faster
   - Connect directly to Power BI for dashboarding (which we'll do next)
   - Practice cloud authentication (service accounts, JSON keys)

# Key Observation

- BigQuery is Google's cloud-based data warehouse, unlike MS SQL Server which runs 
locally on our machine. We used both because MS SQL simulates an operational 
database while BigQuery simulates a cloud analytical warehouse — this is how 
most real companies structure their data.

- To connect Python with BigQuery, we first created a Service Account on Google 
Cloud Platform which generated a JSON key file. This key file was placed in the 
project folder and used via `service_account.Credentials.from_service_account_file()` 
to authenticate Python with BigQuery.

- The following tables were loaded into the `retail_data` dataset in BigQuery using 
`to_gbq()` with the service account credentials:
  - dim_customer, dim_product, dim_date, fact_sales

- All 4 analytical queries were run both in the BigQuery console and from Python 
using `client.query()`. Key business insights discovered:
  - **United Kingdom** generates the most revenue at £14,722,423 — expected since 
  this is a UK-based retailer
  - **Thursday** is the highest revenue day at £3,841,082 and **Saturday** is almost 
  zero at £9,803 — confirming this is a B2B business that operates on weekdays only
  - **November** consistently shows the highest monthly revenue across both years — 
  likely driven by pre-Christmas shopping